In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="kX0RjUlt2tVQzkpKm4ZV")
project = rf.workspace("hemavardhan").project("landmines-detection-dataset-sewvx")
version = project.version(1)
dataset = version.download("coco")


loading Roboflow workspace...
loading Roboflow project...


In [ ]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate model architecture (must match what you trained)
num_classes = 2  # 1 class (landmine) + background
model = fasterrcnn_resnet50_fpn(weights=None)

# Replace the classifier head
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

# Load trained weights
checkpoint_path = "/content/fasterrcnn_landmine_best.pth"
state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)

model.to(device)
model.eval()

print(" Model loaded successfully from:", checkpoint_path)


RuntimeError: PytorchStreamReader failed reading zip archive: invalid header or archive is corrupted

In [ ]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn  # Corrected import
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torchvision import transforms
from PIL import Image
import os
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Initialize model
num_classes = 2  # background + landmine
model = fasterrcnn_resnet50_fpn(weights=None, num_classes=num_classes)
checkpoint_path = "/content/fasterrcnn_landmine_best.pth"

# Load weights
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.to(device)
model.eval()

print("Model loaded successfully.")


RuntimeError: PytorchStreamReader failed reading zip archive: invalid header or archive is corrupted

In [ ]:
# Change these to your actual dataset paths
img_dir = "/content/Landmines-detection-dataset-1/valid"
ann_file = "/content/Landmines-detection-dataset-1/valid/_annotations.coco.json"

coco_gt = COCO(ann_file)
transform = transforms.Compose([transforms.ToTensor()])


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!


In [ ]:
!pip install --upgrade torch torchvision --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 752.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
!pip install torch==2.1.0 torchvision==0.16.0 --quiet


In [ ]:
results = []

for img_id in tqdm(coco_gt.getImgIds(), desc="Evaluating"):
    img_info = coco_gt.loadImgs(img_id)[0]
    img_path = os.path.join(img_dir, img_info['file_name'])
    image = Image.open(img_path).convert("RGB")
    img_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)[0]

    boxes = outputs['boxes'].cpu().numpy()
    scores = outputs['scores'].cpu().numpy()
    labels = outputs['labels'].cpu().numpy()

    for box, score, label in zip(boxes, scores, labels):
        x_min, y_min, x_max, y_max = box
        width, height = x_max - x_min, y_max - y_min
        result = {
            "image_id": img_id,
            "category_id": int(label),
            "bbox": [x_min, y_min, width, height],
            "score": float(score)
        }
        results.append(result)

print("Predictions generated.")


Evaluating:   0%|          | 0/240 [00:08<?, ?it/s]


AttributeError: module 'torchvision' has no attribute '_is_tracing'

In [ ]:
from pycocotools.cocoeval import COCOeval

if len(results) > 0:
    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
else:
    print("⚠️ No predictions to evaluate.")


NameError: name 'val_loader' is not defined